In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os
import subprocess
import shutil
from diffusers import StableDiffusionPipeline, AutoencoderKL
import torch
from PIL import Image
from tqdm import tqdm

# ----- CONFIGURATION -----
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

MODEL_NAME, BASE_MODEL = 'V5.1', "SG161222/Realistic_Vision_V5.1_noVAE"
VAE_MODEL = "stabilityai/sd-vae-ft-mse"

DATA_ROOT = "data_aug/train"
OUTPUT_ROOT = "data_aug/synthetic/synthetic_full_real"
IMAGE_ROOT = "data_aug/synthetic/synthetic_train_realistic"
TRAIN_SCRIPT = "diffusers/examples/dreambooth/train_dreambooth.py"
NUM_IMAGES = 100

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(IMAGE_ROOT, exist_ok=True)

# Ensure GPU is available
assert torch.cuda.is_available(), "CUDA GPU is required"
device = "cuda"

print(f'Using model {MODEL_NAME}')

# ----- LOOP OVER CLASSES -----
class_folders = [f for f in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, f))]

for i, class_name in enumerate(class_folders):
    instance_dir = os.path.join(DATA_ROOT, class_name)
    output_dir = os.path.join(OUTPUT_ROOT, class_name)
    image_dir = os.path.join(IMAGE_ROOT, class_name)
    instance_prompt = f"photo of a <{class_name}> child"

    if os.path.exists(image_dir):
        print(f"[{i+1}/{len(class_folders)}] Skipping {class_name}, already trained and generated.")
        continue

    print(f"[{i+1}/{len(class_folders)}] Training DreamBooth for class: {class_name}")

    # ----- TRAINING -----
    command = f"""
    accelerate launch --main_process_port=29600 {TRAIN_SCRIPT} \
      --pretrained_model_name_or_path="{BASE_MODEL}" \
      --instance_data_dir="{instance_dir}" \
      --output_dir="{output_dir}" \
      --instance_prompt="{instance_prompt}" \
      --resolution=512 \
      --train_batch_size=1 \
      --gradient_accumulation_steps=1 \
      --learning_rate=1e-6 \
      --lr_scheduler="constant" \
      --max_train_steps=800 \
      --checkpointing_steps=800 \
      --mixed_precision="fp16" \
      --train_text_encoder
    """
    subprocess.run(command, shell=True, check=True)

    print(f"Generating images for: {class_name}")

    # ----- LOAD VAE -----
    vae = AutoencoderKL.from_pretrained(
        VAE_MODEL,
        torch_dtype=torch.float16
    ).to(device)
    
    # ----- INFERENCE -----
    pipe = StableDiffusionPipeline.from_pretrained(
        output_dir,
        vae=vae,
        torch_dtype=torch.float16,
        # safety_checker=None
    ).to(device)

    os.makedirs(image_dir, exist_ok=True)

    for n in tqdm(range(NUM_IMAGES), desc=f'Generating {class_name}', ncols=80):
        with torch.autocast("cuda"):
            image = pipe(instance_prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
        image.save(os.path.join(image_dir, f"{class_name}_{n+1}.png"))

    # pipe.to("cpu")
    del pipe
    del vae
    torch.cuda.empty_cache()

    print(f"Deleting model weights for {class_name} to save space.")
    shutil.rmtree(output_dir)

    # Optional: break after one class for testing
    # break

print("All training, generation, and cleanup complete.")
